In [ ]:
# Melakukan cosine similarity berdasarkan query

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from openpyxl.styles import Font, PatternFill, Alignment
import numpy as np
import openpyxl
import re

CSV_PATH    = "year-2026.merged.csv"
OUTPUT_PATH = "search_results_multiple.xlsx"
THRESHOLD   = 0.50

QUERIES = [
    "tiket pesawat",
    "tiket perjalanan udara",
    "tiket penerbangan",
]

# Excel-incompatible characters
ILLEGAL_XML_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

def clean_excel_text(x):
    if isinstance(x, str):
        return ILLEGAL_XML_RE.sub("", x)
    return x

# Step 1 – Load data & clean text columns
df = pd.read_csv(CSV_PATH)
df = df.applymap(clean_excel_text)   # clean all string cells

corpus = df["paket"].fillna("").tolist()

print("Vectorizing corpus…")
vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=100_000,
)
tfidf_matrix = vectorizer.fit_transform(corpus)

seen_indices = set()
result_frames = []

for query in QUERIES:
    query_vec = vectorizer.transform([query])
    scores = (query_vec @ tfidf_matrix.T).toarray().flatten()

    qualifying = np.where(scores >= THRESHOLD)[0]
    new_indices = [i for i in qualifying if i not in seen_indices]

    if not new_indices:
        print(f"  '{query}' → 0 new rows above threshold")
        continue

    new_indices = sorted(new_indices, key=lambda i: scores[i], reverse=True)

    chunk = df.iloc[new_indices].copy().reset_index(drop=True)
    chunk.insert(0, "matched_query", query)
    chunk.insert(0, "similarity_score", scores[new_indices])

    result_frames.append(chunk)
    seen_indices.update(new_indices)

    print(f"  '{query}' → {len(new_indices)} new rows")

if not result_frames:
    print("No rows met the threshold. Exiting.")
    raise SystemExit

results = pd.concat(result_frames, ignore_index=True)
print(f"\nTotal unique rows collected: {len(results)}")

# Clean again before export, just to be safe
results = results.map(clean_excel_text)

results.to_excel(OUTPUT_PATH, index=False, sheet_name="Results")

wb = openpyxl.load_workbook(OUTPUT_PATH)
ws = wb["Results"]

header_fill  = PatternFill("solid", start_color="1F4E79", end_color="1F4E79")
header_font  = Font(name="Arial", bold=True, color="FFFFFF", size=11)
header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = header_align

light_blue = PatternFill("solid", start_color="DCE6F1", end_color="DCE6F1")
white = PatternFill("solid", start_color="FFFFFF", end_color="FFFFFF")

for row_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
    fill = light_blue if row_idx % 2 == 0 else white
    for cell in row:
        cell.fill = fill
        cell.font = Font(name="Arial", size=10)
        cell.alignment = Alignment(vertical="center")

# similarity_score formatting
for cell in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=1):
    for c in cell:
        c.number_format = "0.0000"

ws.freeze_panes = "A2"
for col in ws.columns:
    max_len = max((len(str(cell.value)) for cell in col if cell.value), default=10)
    ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 50)

ws.row_dimensions[1].height = 30
wb.save(OUTPUT_PATH)

print(f"✅ Done!  {len(results)} rows × {len(results.columns)} columns  →  {OUTPUT_PATH}")

C:\Users\Pongo\AppData\Local\Temp\ipykernel_26052\895810744.py:28: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_excel_text)   # clean all string cells


Vectorizing corpus…
  'tiket pesawat' → 2577 new rows
  'tiket perjalanan udara' → 89 new rows
  'tiket penerbangan' → 7 new rows

Total unique rows collected: 2673
✅ Done!  2673 rows × 23 columns  →  search_results_multiple.xlsx


In [ ]:
# Melakukan Penghapusan baris yang memiliki label "beda" setelah dilakukan validasi manual

In [4]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
import re

INPUT_PATH  = "search_results_multiple.xlsx"
OUTPUT_PATH = "search_results_filtered.xlsx"

ILLEGAL_XML_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

def clean_excel_text(x):
    if isinstance(x, str):
        return ILLEGAL_XML_RE.sub("", x)
    return x

df = pd.read_excel(INPUT_PATH)
print(f"Rows before: {len(df)}")

df_filtered = df[df["Label"].str.strip() != "beda"].reset_index(drop=True)
print(f"Rows after:  {len(df_filtered)}")

df_filtered = df_filtered.map(clean_excel_text)
df_filtered.to_excel(OUTPUT_PATH, index=False, sheet_name="Results")

wb = openpyxl.load_workbook(OUTPUT_PATH)
ws = wb["Results"]

header_fill  = PatternFill("solid", start_color="1F4E79", end_color="1F4E79")
header_font  = Font(name="Arial", bold=True, color="FFFFFF", size=11)
header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = header_align

light_blue = PatternFill("solid", start_color="DCE6F1", end_color="DCE6F1")
white      = PatternFill("solid", start_color="FFFFFF", end_color="FFFFFF")

for row_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
    fill = light_blue if row_idx % 2 == 0 else white
    for cell in row:
        cell.fill = fill
        cell.font = Font(name="Arial", size=10)
        cell.alignment = Alignment(vertical="center")

# similarity_score formatting (column 1)
for cell in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=1):
    for c in cell:
        c.number_format = "0.0000"

ws.freeze_panes = "A2"
for col in ws.columns:
    max_len = max((len(str(cell.value)) for cell in col if cell.value), default=10)
    ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 50)

ws.row_dimensions[1].height = 30
wb.save(OUTPUT_PATH)

print(f"✅ Done!  {len(df_filtered)} rows × {len(df_filtered.columns)} columns  →  {OUTPUT_PATH}")

Rows before: 2673
Rows after:  2473
✅ Done!  2473 rows × 24 columns  →  search_results_filtered.xlsx


In [ ]:
# Melakukan klasifikasi berdasarkan 3 kolom yang dipilih (disini saya menggunakan kolom "paket", "Pagu", "UMKM"

In [9]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from openpyxl.styles import Font, PatternFill, Alignment
import numpy as np
import openpyxl
import re

INPUT_PATH  = "search_results_filtered.xlsx"
OUTPUT_PATH = "sorted_similarity_results.xlsx"

TARGET_ROW_INDEX = 0

W_PAKET  = 0.60
W_PAGU   = 0.30
W_UMKM   = 0.10

ILLEGAL_XML_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

def clean_excel_text(x):
    if isinstance(x, str):
        return ILLEGAL_XML_RE.sub("", x)
    return x

def normalize_bool(x):
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return False
    s = str(x).strip().lower()
    return s in {"1", "true", "yes", "y", "umkm", "t"}

# Load and clean
df = pd.read_excel(INPUT_PATH)
df = df.map(clean_excel_text)

if TARGET_ROW_INDEX < 0 or TARGET_ROW_INDEX >= len(df):
    raise IndexError(f"TARGET_ROW_INDEX must be between 0 and {len(df)-1}")

df = df.reset_index(drop=True)
df["source_row"] = df.index

target = df.iloc[TARGET_ROW_INDEX]

print("Vectorizing paket text…")
paket_corpus = df["paket"].fillna("").astype(str).tolist()
target_paket = str(target["paket"])

vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    max_features=100_000,
)

tfidf_matrix = vectorizer.fit_transform(paket_corpus)
target_vec = vectorizer.transform([target_paket])
paket_sim = (target_vec @ tfidf_matrix.T).toarray().flatten()

# pagu similarity
pagu_values = pd.to_numeric(df["pagu"], errors="coerce").fillna(0).astype(float)
target_pagu = float(pd.to_numeric(pd.Series([target["pagu"]]), errors="coerce").fillna(0).iloc[0])

pagu_log = np.log1p(pagu_values)
target_pagu_log = np.log1p(target_pagu)

all_pagu = np.append(pagu_log.values, target_pagu_log)
scaler = MinMaxScaler()
all_pagu_scaled = scaler.fit_transform(all_pagu.reshape(-1, 1)).ravel()

pagu_scaled = all_pagu_scaled[:-1]
target_pagu_scaled = all_pagu_scaled[-1]
pagu_sim = 1 - np.abs(pagu_scaled - target_pagu_scaled)

# isUMKM similarity
umkm_target = normalize_bool(target["isUMKM"])
umkm_sim = df["isUMKM"].apply(normalize_bool).astype(int).eq(int(umkm_target)).astype(int)

# Final weighted score
df["paket_sim"] = paket_sim
df["pagu_sim"] = pagu_sim
df["umkm_sim"] = umkm_sim

df["similarity_score"] = (
    W_PAKET * df["paket_sim"] +
    W_PAGU * df["pagu_sim"] +
    W_UMKM * df["umkm_sim"]
)

# Remove the target row itself
results = df[df["source_row"] != TARGET_ROW_INDEX].copy()

# Sort by score
results = results.sort_values("similarity_score", ascending=False).reset_index(drop=True)

# Clean again before export
results = results.map(clean_excel_text)

# Save to Excel
results.to_excel(OUTPUT_PATH, index=False, sheet_name="Results")

# Styling
wb = openpyxl.load_workbook(OUTPUT_PATH)
ws = wb["Results"]

header_fill  = PatternFill("solid", start_color="1F4E79", end_color="1F4E79")
header_font  = Font(name="Arial", bold=True, color="FFFFFF", size=11)
header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = header_align

light_blue = PatternFill("solid", start_color="DCE6F1", end_color="DCE6F1")
white = PatternFill("solid", start_color="FFFFFF", end_color="FFFFFF")

for row_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
    fill = light_blue if row_idx % 2 == 0 else white
    for cell in row:
        cell.fill = fill
        cell.font = Font(name="Arial", size=10)
        cell.alignment = Alignment(vertical="center")

# Format similarity_score column
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=1):
    for c in row:
        c.number_format = "0.0000"

ws.freeze_panes = "A2"

for col in ws.columns:
    max_len = max((len(str(cell.value)) for cell in col if cell.value is not None), default=10)
    ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 50)

ws.row_dimensions[1].height = 30
wb.save(OUTPUT_PATH)

print(f"✅ Done!  {len(results)} rows × {len(results.columns)} columns  →  {OUTPUT_PATH}")

Vectorizing paket text…
✅ Done!  2472 rows × 28 columns  →  sorted_similarity_results.xlsx
